# GLD_BUILD_FACT_TRANSACTIONS
**Layer:** Gold  
**Purpose:** Join Silver transaction logs with Silver dimension tables to build the Gold star-schema fact table `gld_fact_transaction_logs`.  Applies Z-score anomaly scoring on `response_time_ms` and `amount` to set `is_anomaly`, `anomaly_score`, and `ai_risk_label`.  
**Pattern:** Full-partition refresh for the processed date range; MERGE into Gold Delta on `txn_log_sk`.

## 1. Parameters

In [ ]:
batch_id            = "dev-run-00000000"
storage_account     = "adlsbankingdev"
silver_container    = "silver"
gold_container      = "gold"
keyvault_name       = "kv-banking-dev"
watermark_date      = "2024-01-01"
run_date            = "2024-01-02"
# Anomaly thresholds (Z-score based)
z_score_suspicious  = 2.0    # |Z| >= 2.0  -> SUSPICIOUS
z_score_high_risk   = 3.5    # |Z| >= 3.5  -> HIGH_RISK

## 2. Imports and Spark Configuration

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    LongType, BooleanType, DoubleType, DecimalType
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable
import datetime
import hashlib

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "128")
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")
spark.conf.set("spark.sql.adaptive.enabled", "true")

def adls_path(container, *parts):
    base = f"abfss://{container}@{storage_account}.dfs.core.windows.net"
    return "/".join([base] + list(parts))

ingestion_timestamp = datetime.datetime.utcnow().isoformat() + "Z"
silver_delta_base   = adls_path(silver_container, "delta")
gold_delta_base     = adls_path(gold_container,   "delta")
print(f"batch_id={batch_id}  window=[{watermark_date}, {run_date})")
print(f"Silver base: {silver_delta_base}")
print(f"Gold base  : {gold_delta_base}")

## 3. Read Silver Transaction Logs

In [ ]:
slv_txn = (
    spark.read.format("delta")
    .load(f"{silver_delta_base}/slv_transaction_logs")
    .filter(
        (F.col("event_date_sk") >= F.lit(watermark_date.replace("-", "")).cast(IntegerType())) &
        (F.col("event_date_sk") <  F.lit(run_date.replace("-", "")).cast(IntegerType()))
    )
)
print(f"Silver transaction records: {slv_txn.count():,}")

## 4. Read Silver Dimension Tables

In [ ]:
dim_account = (
    spark.read.format("delta")
    .load(f"{silver_delta_base}/slv_dim_account")
    .select("account_id", "account_sk", "customer_sk", "product_type")
)

dim_channel = (
    spark.read.format("delta")
    .load(f"{silver_delta_base}/slv_dim_channel")
    .select("channel_code", "channel_sk", "channel_name")
)

dim_txn_type = (
    spark.read.format("delta")
    .load(f"{silver_delta_base}/slv_dim_transaction_type")
    .select("txn_type_code", "txn_type_sk", "txn_category")
)

dim_service = (
    spark.read.format("delta")
    .load(f"{silver_delta_base}/slv_dim_service")
    .select("service_code", "service_sk", "service_name")
)

dim_date = (
    spark.read.format("delta")
    .load(f"{silver_delta_base}/slv_dim_date")
    .select("date_sk", "calendar_date", "year", "month", "day", "quarter")
)

print("Dimension tables loaded.")

## 5. Build Gold Fact — Star Schema Join

In [ ]:
# Broadcast all dimension tables (small lookup tables)
fact_base = (
    slv_txn
    .join(F.broadcast(dim_account),   on="account_id",    how="left")
    .join(F.broadcast(dim_channel),   on="channel_code",  how="left")
    .join(F.broadcast(dim_txn_type),  on="txn_type_code", how="left")
    .join(F.broadcast(dim_service),   on="service_code",  how="left")
    .join(F.broadcast(dim_date),      slv_txn.event_date_sk == dim_date.date_sk, how="left")
    .select(
        # Dimension surrogate keys
        F.col("account_sk"),
        F.col("customer_sk"),
        F.col("channel_sk"),
        F.col("txn_type_sk"),
        F.col("service_sk"),
        F.col("event_date_sk"),
        # Degenerate dimensions / measures
        F.col("raw_log_id"),
        F.col("event_timestamp"),
        F.col("transaction_amount").alias("amount"),
        F.col("currency_code"),
        F.col("status_code"),
        F.col("response_time_ms"),
        F.col("error_code"),
        F.col("merchant_id"),
        # Audit
        F.col("slv_batch_id"),
    )
)
print(f"Fact base row count: {fact_base.count():,}")

## 6. Anomaly Scoring — Z-Score on amount and response_time_ms

In [ ]:
# ── Compute global mean and stddev for the batch window ──────────────────────
# Z-score = (value - mean) / stddev
# Combined anomaly score = max(|Z_amount|, |Z_response_time|) normalised to [0, 1]
# via a logistic-style cap:  score = min(max_z, 5.0) / 5.0

stats = fact_base.select(
    F.mean(F.col("amount").cast(DoubleType())).alias("mean_amount"),
    F.stddev_pop(F.col("amount").cast(DoubleType())).alias("std_amount"),
    F.mean(F.col("response_time_ms").cast(DoubleType())).alias("mean_rt"),
    F.stddev_pop(F.col("response_time_ms").cast(DoubleType())).alias("std_rt"),
).collect()[0]

mean_amount = float(stats["mean_amount"]) if stats["mean_amount"] is not None else 0.0
std_amount  = float(stats["std_amount"])  if stats["std_amount"]  and float(stats["std_amount"]) > 0 else 1.0
mean_rt     = float(stats["mean_rt"])     if stats["mean_rt"]     is not None else 0.0
std_rt      = float(stats["std_rt"])      if stats["std_rt"]      and float(stats["std_rt"]) > 0 else 1.0

print(f"amount  : mean={mean_amount:.4f}  std={std_amount:.4f}")
print(f"resp_ms : mean={mean_rt:.4f}  std={std_rt:.4f}")

scored_df = (
    fact_base
    .withColumn("z_amount",
        F.abs((F.col("amount").cast(DoubleType()) - F.lit(mean_amount)) / F.lit(std_amount)))
    .withColumn("z_rt",
        F.abs((F.col("response_time_ms").cast(DoubleType()) - F.lit(mean_rt)) / F.lit(std_rt)))
    # Combined max Z-score
    .withColumn("max_z", F.greatest(F.col("z_amount"), F.col("z_rt")))
    # Normalise to [0, 1]: cap at Z=5 then divide
    .withColumn("anomaly_score",
        F.least(F.col("max_z"), F.lit(5.0)) / F.lit(5.0))
    # Risk label based on raw Z thresholds
    .withColumn("ai_risk_label",
        F.when(F.col("max_z") >= F.lit(z_score_high_risk),   F.lit("HIGH_RISK"))
         .when(F.col("max_z") >= F.lit(z_score_suspicious),  F.lit("SUSPICIOUS"))
         .otherwise(F.lit("NORMAL")))
    .withColumn("is_anomaly",
        (F.col("ai_risk_label") != F.lit("NORMAL")))
    .drop("z_amount", "z_rt", "max_z")
)

anomaly_summary = scored_df.groupBy("ai_risk_label").count().collect()
for row in anomaly_summary:
    print(f"  {row['ai_risk_label']:12s}: {row['count']:,}")

## 7. Generate Surrogate Key txn_log_sk

In [ ]:
# Deterministic surrogate key: SHA-256 of raw_log_id, truncated to 18 digits
# This ensures idempotency across re-runs.
@F.udf(returnType=LongType())
def make_txn_log_sk(raw_log_id: str) -> int:
    if raw_log_id is None:
        return -1
    h = hashlib.sha256(raw_log_id.encode()).hexdigest()
    return int(h[:15], 16) % (10 ** 18)

final_df = (
    scored_df
    .withColumn("txn_log_sk",               make_txn_log_sk(F.col("raw_log_id")))
    .withColumn("gld_batch_id",             F.lit(batch_id))
    .withColumn("gld_ingestion_timestamp",  F.lit(ingestion_timestamp))
    .withColumn("anomaly_score",            F.col("anomaly_score").cast(DoubleType()))
    .withColumn("is_anomaly",               F.col("is_anomaly").cast(BooleanType()))
)
print(f"Final Gold fact rows: {final_df.count():,}")

## 8. Upsert to Gold Delta Table (MERGE on txn_log_sk)

In [ ]:
gold_table_path = f"{gold_delta_base}/gld_fact_transaction_logs"

if not DeltaTable.isDeltaTable(spark, gold_table_path):
    print("Gold fact table not found — creating from current batch.")
    (
        final_df.write
        .format("delta")
        .mode("overwrite")
        .partitionBy("event_date_sk")
        .save(gold_table_path)
    )
    print(f"Created: {gold_table_path}")
else:
    gold_delta = DeltaTable.forPath(spark, gold_table_path)
    (
        gold_delta.alias("target")
        .merge(
            final_df.alias("source"),
            "target.txn_log_sk = source.txn_log_sk"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    metrics = gold_delta.history(1).select("operationMetrics").collect()[0][0]
    print(f"MERGE complete. Metrics: {metrics}")
    print(f"Gold fact build finished: batch_id={batch_id}")